# Score-Based Diffusion Models and the Connection to Neural ODEs

This notebook focuses on the **continuous-time theory** behind score-based diffusion models. The goal is to explain how diffusion can be written as a stochastic differential equation (SDE), why learning the **score** is enough for generation, and how the same stochastic process induces a deterministic **probability flow ODE** that connects naturally to the Neural ODE viewpoint.

Unlike the introductory diffusion notebook, this notebook is theory-only: no toy examples, no training code, and no sampling implementation. The emphasis is on the mathematical structure.


## 1. The score function

For a probability density $p(\mathbf{x})$, the **score** is the gradient of the log density:

$$
\nabla_{\mathbf{x}} \log p(\mathbf{x}).
$$

This vector field points in the direction of increasing probability density. Intuitively:

- where the density is large, the score tends to point toward high-probability regions,
- where the density changes rapidly, the score has large magnitude,
- if we know the score everywhere, we know the local geometry of the distribution even if we do not know the normalizing constant of $p(\mathbf{x})$.

That last point is central. Many generative models struggle with normalized densities, but score-based methods only need derivatives of the log density with respect to the state.


## 2. Forward diffusion as an SDE

In score-based diffusion, the discrete noising process is replaced by a **continuous-time stochastic differential equation**:

$$
\mathrm{d}\mathbf{x} = \mathbf{f}(\mathbf{x}, t)\,\mathrm{d}t + g(t)\,\mathrm{d}\mathbf{w},
$$

where:

- $\mathbf{f}(\mathbf{x}, t)$ is the drift,
- $g(t)$ is the diffusion strength,
- $\mathbf{w}$ is standard Brownian motion.

The forward SDE gradually perturbs the data distribution $p_0(\mathbf{x})$ into a simple reference distribution, usually Gaussian, by time $T$.

Two common choices are:

1. **Variance preserving (VP) SDE**

$$
\mathrm{d}\mathbf{x} = -\tfrac{1}{2}\beta(t)\mathbf{x}\,\mathrm{d}t + \sqrt{\beta(t)}\,\mathrm{d}\mathbf{w}.
$$

This is the continuous-time analogue of the DDPM forward process.

2. **Variance exploding (VE) SDE**

$$
\mathrm{d}\mathbf{x} = \sqrt{\frac{\mathrm{d}[\sigma^2(t)]}{\mathrm{d}t}}\,\mathrm{d}\mathbf{w}.
$$

Here the mean stays fixed while the variance grows over time.

In both cases, the purpose of the forward process is the same: transform a complicated data distribution into one that is easy to sample.


## 3. Time-dependent marginals and perturbation kernels

The forward SDE defines a family of intermediate distributions $p_t(\mathbf{x})$ for $t \in [0, T]$. These are the distributions of the random state after evolving the forward process for time $t$.

A useful viewpoint is to think of the forward process as repeatedly applying a **noise perturbation kernel**:

$$
p_t(\mathbf{x}_t) = \int p_{0t}(\mathbf{x}_t \mid \mathbf{x}_0) p_0(\mathbf{x}_0)\,\mathrm{d}\mathbf{x}_0.
$$

Even if $p_0$ is complicated, the conditional perturbation kernel $p_{0t}(\mathbf{x}_t \mid \mathbf{x}_0)$ is often Gaussian and analytically tractable. This lets us sample noisy states at arbitrary times during training.

The learning target is not the score of the clean data distribution directly, but the **time-dependent score**:

$$
\nabla_{\mathbf{x}} \log p_t(\mathbf{x}).
$$

As $t$ increases, the distribution becomes smoother and easier to model. The full generative problem is then handled by learning the score field across all times.


## 4. Reverse-time SDE

The key theoretical result is that the time-reversal of a diffusion process is itself a diffusion process. To state this carefully, start from the forward SDE

$$
\mathrm{d}\mathbf{x} = \mathbf{f}(\mathbf{x}, t)\,\mathrm{d}t + g(t)\,\mathrm{d}\mathbf{w},
$$

where:

- $\mathbf{x}(t) \in \mathbb{R}^d$ is the random state at time $t$,
- $\mathbf{f}(\mathbf{x}, t)$ is the drift term,
- $g(t)$ is the scalar diffusion amplitude,
- $\mathbf{w}(t)$ is standard $d$-dimensional Brownian motion,
- $p_t(\mathbf{x})$ is the probability density of the random variable $\mathbf{x}(t)$.

The forward process moves from the data distribution at $t=0$ toward a simple noise distribution at $t=T$. The reverse problem asks for dynamics that move samples from the final noisy density $p_T$ back through the same family of intermediate marginals until we recover $p_0$.

To describe that backward evolution, introduce a reverse time variable. One common notation is to write the reverse process in terms of a decreasing time parameter, often denoted $\bar{t}$, meaning that trajectories are integrated from $T$ down to $0$. With that convention, the reverse-time SDE is

$$
\mathrm{d}\mathbf{x} = \left[\mathbf{f}(\mathbf{x}, t) - g(t)^2 \nabla_{\mathbf{x}} \log p_t(\mathbf{x})\right] \mathrm{d}\bar{t} + g(t)\,\mathrm{d}\bar{\mathbf{w}},
$$

where:

- $\nabla_{\mathbf{x}} \log p_t(\mathbf{x})$ is the **score** of the time-$t$ marginal,
- $\bar{\mathbf{w}}$ is Brownian motion for the reverse-time process,
- $\mathrm{d}\bar{t}$ indicates that the process is run backward from $T$ to $0$.

The important change is the extra drift correction

$$
- g(t)^2 \nabla_{\mathbf{x}} \log p_t(\mathbf{x})
$$

which does not appear in the forward SDE. Intuitively, the forward process spreads probability mass outward by diffusion. To reverse that spreading, the backward dynamics need information about how probability mass is distributed locally at time $t$. That information is exactly encoded by the score.

A useful way to read the reverse drift is:

- $\mathbf{f}(\mathbf{x}, t)$ keeps the original deterministic part of the dynamics,
- $-g(t)^2 \nabla_{\mathbf{x}} \log p_t(\mathbf{x})$ pulls states back toward regions where the time-$t$ density is larger.

So the reverse-time SDE is not obtained by simply changing the sign of time in the forward SDE. Reversing diffusion requires an additional score term because stochastic spreading is not undone by naive time reversal alone.

If we knew the exact time-dependent score for every $t$, then we could simulate this reverse SDE starting from a sample at time $T$ and obtain a sample from the data distribution at time $0$.

So the generative modeling problem becomes: learn an accurate approximation

$$
\mathbf{s}_\theta(\mathbf{x}, t) \approx \nabla_{\mathbf{x}} \log p_t(\mathbf{x}).
$$


## 5. Learning the score

Directly learning $\nabla_{\mathbf{x}} \log p_t(\mathbf{x})$ seems difficult because $p_t$ is unknown. Score-based diffusion gets around this using **score matching** and, more specifically, **denoising score matching**.

The model is trained on noisy samples produced by the perturbation kernel. For many diffusion constructions, the conditional score of the perturbation kernel is available analytically:

$$
\nabla_{\mathbf{x}_t} \log p_{0t}(\mathbf{x}_t \mid \mathbf{x}_0).
$$

Then the network $\mathbf{s}_\theta(\mathbf{x}_t, t)$ is trained to match this target in expectation over clean data and noise. A standard continuous-time objective is

$$
\mathcal{L}(\theta) = \mathbb{E}_{t, \mathbf{x}_0, \mathbf{x}_t}\left[\lambda(t) \left\| \mathbf{s}_\theta(\mathbf{x}_t, t) - \nabla_{\mathbf{x}_t} \log p_{0t}(\mathbf{x}_t \mid \mathbf{x}_0) \right\|^2\right],
$$

with a weighting function $\lambda(t)$ to balance learning across noise scales.

This is the continuous-time score-based analogue of the discrete diffusion objective. In DDPM language, predicting the score, predicting the noise, and predicting the clean sample are closely related parameterizations of the same denoising information.


## 6. Distribution evolution and the Fokker-Planck equation

An SDE describes individual sample paths, but it also induces an evolution equation for the density itself. If

$$
\mathrm{d}\mathbf{x} = \mathbf{f}(\mathbf{x}, t)\,\mathrm{d}t + g(t)\,\mathrm{d}\mathbf{w},
$$

then the marginal density evolves according to the **Fokker-Planck equation**:

$$
\frac{\partial p_t}{\partial t} = -\nabla \cdot (\mathbf{f} p_t) + \frac{1}{2} g(t)^2 \Delta p_t.
$$

This equation makes the roles of drift and diffusion explicit:

- the drift term transports mass,
- the diffusion term smooths the density.

The reverse-time SDE can be understood as the stochastic dynamics whose density evolution in reverse matches the forward process. The score appears because reversing diffusion requires knowledge of how probability mass is locally arranged.


## 7. The probability flow ODE

A particularly elegant result is that every forward diffusion SDE has an associated **probability flow ODE**:

$$
\frac{\mathrm{d}\mathbf{x}}{\mathrm{d}t} = \mathbf{f}(\mathbf{x}, t) - \frac{1}{2} g(t)^2 \nabla_{\mathbf{x}} \log p_t(\mathbf{x}).
$$

This ODE is deterministic, yet it shares the **same marginal densities** $p_t(\mathbf{x})$ as the original SDE. That is the key statement: stochastic diffusion sampling and deterministic ODE transport describe two different pathwise dynamics for the same evolving family of distributions.

Replacing the true score with a learned score model gives the generative ODE:

$$
\frac{\mathrm{d}\mathbf{x}}{\mathrm{d}t} = \mathbf{f}(\mathbf{x}, t) - \frac{1}{2} g(t)^2 \mathbf{s}_\theta(\mathbf{x}, t).
$$

This is one of the cleanest bridges between score-based diffusion and continuous-depth models.


## 8. Connection to Neural ODEs

A Neural ODE defines a continuous-time transformation of the state through

$$
\frac{\mathrm{d}\mathbf{x}}{\mathrm{d}t} = \mathbf{v}_\theta(\mathbf{x}, t),
$$

where $\mathbf{v}_\theta$ is a learned vector field. The probability flow ODE has exactly this form, with

$$
\mathbf{v}_\theta(\mathbf{x}, t) = \mathbf{f}(\mathbf{x}, t) - \frac{1}{2} g(t)^2 \mathbf{s}_\theta(\mathbf{x}, t).
$$

So from a dynamical-systems point of view:

- the forward diffusion defines a family of distributions over time,
- the learned score provides the missing geometric information about those distributions,
- the probability flow ODE turns that geometry into a deterministic transport map.

This is why score-based generative modeling can be viewed as a **Neural ODE with a very specific vector field structure**. The vector field is not learned arbitrarily; it is constrained by the forward SDE and by score estimation.

There is also a practical consequence. Once a generative process is written as an ODE, we can use standard ODE solvers to sample trajectories, control numerical error, and sometimes compute exact or tractable likelihoods via the change-of-variables formula.


## 9. Why the ODE view is useful

The SDE and ODE formulations generate from the same marginals, but they emphasize different things.

The **reverse SDE** view is natural when we think of diffusion as iterative denoising with stochasticity. It often gives robust samplers and makes the connection to noise injection explicit.

The **probability flow ODE** view is useful because deterministic dynamics let us borrow machinery from continuous normalizing flows and Neural ODEs. In particular, if the dynamics are

$$
\frac{\mathrm{d}\mathbf{x}}{\mathrm{d}t} = \mathbf{v}_\theta(\mathbf{x}, t),
$$

then the log density evolves as

$$
\frac{\mathrm{d}}{\mathrm{d}t} \log p_t(\mathbf{x}(t)) = - \nabla \cdot \mathbf{v}_\theta(\mathbf{x}(t), t).
$$

This gives a route to likelihood estimation that is much harder to access from the stochastic sampler alone.

Conceptually, the ODE view says that diffusion generation is not only about noise removal. It is also a continuous transport problem in state space.


## 10. The role of numerical solvers

Once we formulate the model as either a reverse-time SDE or a probability flow ODE, generation becomes a **numerical integration problem**.

For the reverse SDE, we use an SDE discretization scheme, such as Euler-Maruyama or predictor-corrector methods. These approximate both the deterministic drift and the stochastic Brownian increments.

For the probability flow ODE, we can instead use deterministic ODE solvers such as Euler, Heun, RK4, or adaptive solvers. This is the direct connection to Neural ODE practice: once the learned score defines the vector field, the sample path is obtained by integrating that vector field backward in time.

The solver choice matters because the learned score is only approximate. In practice, changing the solver changes the quality-speed tradeoff, just as in Neural ODEs and other continuous-depth models.

So the connection is not only conceptual. It is computational:

- score model + reverse SDE gives a stochastic sampler,
- score model + probability flow ODE gives a deterministic continuous-depth sampler,
- both rely on the same learned score field.


## 11. Summary

The main ideas are:

- a score-based diffusion model learns the time-dependent score $\nabla_{\mathbf{x}} \log p_t(\mathbf{x})$,
- the forward process is naturally written as an SDE that gradually turns data into noise,
- the reverse-time SDE uses the score to turn noise back into samples,
- the same family of marginals can also be generated by a deterministic probability flow ODE,
- that ODE has the same structure as a Neural ODE, with a vector field defined by the forward drift and the learned score.

This is the conceptual bridge: **score-based diffusion models are stochastic generative models, but they also induce deterministic continuous-time dynamics that fit naturally into the Neural ODE framework.**
